# 01 — Build the dataset: inputs, outputs, and the on-disk cache

Define the **data contract** for the model and materialize it to a fast cache that
`model_results.ipynb` trains on. Written to be *read*: each step prints the shape/dtype
it produces, so the preprocessing is fully legible.

**Learning problem (v1).** For each calendar day **D** *defined in CDT (UTC−5)* with ≥1
observed **flood**, predict **which 50 km CONUS-land cells flood on D** from the
**previous day's (D−1)** GOES imagery. One sample = one `(D−1 imagery → D flood map)`
pair, over **2019 and 2020**.

> **Labels = observed floods: Groundsource ∪ NCEI storm events** (not NWS warnings).
> Groundsource = flood extents from local news reports (polygons); storm events = NCEI
> confirmed flood footprints. A cell is positive on D if either source places a flood there.

> **Days are CDT; GOES files stay UTC.** Each event's day(s) are CDT (UTC−5): **storm
> events** (real UTC times) shift −5 h; **Groundsource** dates are already day-resolution so
> they're used as-is. The input is the GOES frames belonging to **CDT day D−1** — frames
> whose scan time (UTC−5) falls on D−1. Because the imagery spans the full day (8 frames at
> 00, 03, …, 21 UTC), a single CDT day's 8 frames straddle **two** UTC folders: the
> **06–21 UTC** frames of day D−1 plus the **00 & 03 UTC** frames of day D. An event active
> across **up to 5** CDT days counts on each; longer collapses to a **single** (issue-day)
> event.

**The data contract**

| tensor | shape | dtype | meaning |
|---|---|---|---|
| `x` GOES | `(T=8, 5, 1500, 2500)` | f16 | 8 frames (CDT day D−1) × 5 emissive-IR bands on the 2 km grid |
| `t` lead | `(8,)` | f32 | per-frame lead time (whole hours before D 00:00 CDT = D 05:00 UTC) |
| `y` label | `(GRID_R, GRID_C)` | uint8 | 0/1 observed flood per 50 km cell, scored on land only |

> **Bands** = `config.BANDS` = `(8, 10, 11, 14, 15)` — all **emissive IR** (brightness
> temperature), so they carry a real signal in the night frames too (reflectance bands
> would be dark). **Lead time** is the *optional* `t` channel: it is **always cached**, and
> `config.USE_TIME` toggles whether notebook 02 appends it as a 6th input channel — flip it
> without rebuilding `x`. **Location** is added in the model as a learned per-cell embedding
> (not here). **GLM lightning** is skipped for now.

Pipeline: build the **output grid** → rasterize **flood labels** → read & normalize
**GOES** → assemble one **sample** → index all valid samples → **materialize the cache**.

## 0. Config & imports

All knobs live in the repo-root `config.py` (single source of truth). The lever that
matters most is `CELL_KM` (output cell size) — everything downstream derives from it.

In [ ]:
import json
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd

# all shared constants live in the repo-root config.py (single source of truth)
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import (BANDS, CACHE_DIR, CELL_KM, DATA_DIR, IMG_H, IMG_W, N_BAND,
                    N_CH, SPLIT_FRACS, SPLIT_SEED, STATS_PATH, T_FRAMES, USE_TIME,
                    UNIFIED_PARQUET, YEARS, build_grid_cells)

# inputs: 8 full-day GOES frames (5 emissive-IR bands) + an OPTIONAL per-frame lead-time
# channel; output: the next-day flood map on the CELL_KM grid
print(f"GOES seq : (T={T_FRAMES}, {N_BAND}, {IMG_H}, {IMG_W})  "
      f"= {T_FRAMES*N_BAND*IMG_H*IMG_W*2/1e6:.0f} MB f16 (cached)")
lead_state = ("ON — appended as a model channel" if USE_TIME
              else "OFF — cached only, not fed to the model")
print(f"lead time: (T={T_FRAMES},) f32 | USE_TIME={USE_TIME} ({lead_state}) -> N_CH={N_CH}")
print(f"output   : {CELL_KM} km cells | bands {BANDS} | years {YEARS}")

## 1. Output target — the 50 km cell grid

The label space is a fixed grid of square **50 km** cells over CONUS land, generated on
the fly by `config.build_grid_cells()` in an equal-area projection (EPSG:5070) and kept
where they intersect land. Nothing is stored — it's a pure function of `CELL_KM` + the
CONUS boundary, so notebook 02 rebuilds the identical grid. Every label `y` is a
`(GRID_R, GRID_C)` image; the loss is scored only on the `land_mask` cells.

> **Orientation:** row 0 = north, col 0 = west — matching the GOES imagery, so inputs and
> labels share one frame and plots render right-way-up.

In [ ]:
# generated fresh at CELL_KM over CONUS land (config.build_grid_cells; nothing stored)
cells, GRID_R, GRID_C, land_mask = build_grid_cells()

print(f"{CELL_KM} km grid: ({GRID_R}, {GRID_C})  <- every y map is this shape")
print(f"land cells: {int(land_mask.sum())} of {GRID_R*GRID_C}  <- loss computed here")

plt.figure(figsize=(10, 8))
plt.imshow(land_mask, cmap="Greys", interpolation="none")
plt.title(f"land mask  ({GRID_R}x{GRID_C}, {int(land_mask.sum())} cells)")
plt.xlabel("col (W->E)"); plt.ylabel("row (N->S)"); plt.tight_layout()

## 2. Labels — observed floods → daily 0/1 cell maps

Rasterize **observed flood footprints** onto the grid: each event's geometry marks every
50 km cell it **intersects** with a 1 → `labels[day] → (GRID_R, GRID_C)`.

> **Label source = observed floods: Groundsource ∪ NCEI storm events.** Groundsource =
> flood extents extracted from local news reports (polygons); storm events = NCEI confirmed
> flood footprints. A cell is positive on day D if **either** source places a flood there.
> (Not the NWS *warnings* — those are issued predictions, not observations.) Edit
> `LABEL_SOURCES` to switch targets (the choice materially changes the task).

Day-assignment rules:
- **Events are placed on CDT calendar days.** **Storm events** carry a real UTC time, so
  they shift **−5 h to CDT**; **Groundsource** dates are already day-resolution
  (news-report day) so they're used **as-is** (a −5 h shift would wrongly bump every one
  back a day). GOES is *not* shifted — its D−1 frames are selected by CDT date in step 3.
- **Multi-day events split per day, capped at 5.** An event active over **≤
  `MAX_SPLIT_DAYS` (5)** CDT days marks its cells on **every** such day (becoming that many
  one-day events); a longer event collapses to a **single** event on its issue day, so one
  footprint can't smear across weeks of labels.

In [ ]:
# label source: OBSERVED floods = Groundsource (flood extents from local news reports) +
# NCEI storm events (confirmed flood footprints) — NOT the NWS warnings (issued predictions).
# Switch the target by editing this set, e.g. {"ff_warning", "fa_warning"} for warnings.
LABEL_SOURCES = {"groundsource", "storm_event"}
CDT = pd.Timedelta(hours=5)        # storm events are UTC -> CDT (UTC-5); groundsource is day-res
MAX_SPLIT_DAYS = 5                 # <=5 CDT days -> per-day events; longer -> issue day only

u = gpd.read_parquet(UNIFIED_PARQUET)
w = u[u["issue_date"].dt.year.isin(YEARS) & u["source"].isin(LABEL_SOURCES)].copy()

# Place each event on CDT calendar day(s). Only STORM events carry a real UTC time, so only
# they shift -5 h; GROUNDSOURCE dates are already day-resolution (news-report day), so they
# are used as-is (shifting them -5 h would wrongly bump every event back a day). GOES is not
# shifted -- its D-1 frames are selected by CDT date in step 3.
shift = pd.Series(pd.Timedelta(0), index=w.index)
shift[w["source"] == "storm_event"] = CDT
w["issue_cdt"] = w["issue_date"] - shift
w["expire_cdt"] = w["expire_date"] - shift


def event_days(issue, expire):
    "CDT days an event labels: each day if it spans <=MAX_SPLIT_DAYS, else the issue day."
    days = pd.date_range(issue.normalize(), expire.normalize(), freq="D")
    return days if len(days) <= MAX_SPLIT_DAYS else days[:1]


w["days"] = [event_days(i, e) for i, e in zip(w["issue_cdt"], w["expire_cdt"])]
ev = w.explode("days", ignore_index=True).rename(columns={"days": "label_day"})
ev = ev[ev["label_day"].dt.year.isin(YEARS)]          # GOES cache only covers YEARS

# spatial join: which 50 km cell does each (event, day) footprint touch?
j = gpd.sjoin(cells, ev[["label_day", "geometry"]], predicate="intersects")

labels = {}
for day, g in j.groupby("label_day"):
    a = np.zeros((GRID_R, GRID_C), dtype=np.float32)
    a[g["R"], g["C"]] = 1.0
    labels[day.date()] = a

span = (w["expire_cdt"].dt.normalize() - w["issue_cdt"].dt.normalize()).dt.days + 1
pos_per_day = np.array([a[land_mask].sum() for a in labels.values()])
cnt = w["source"].value_counts()
print(f"{len(w):,} flood events ({' + '.join(f'{v:,} {k}' for k, v in cnt.items())}) "
      f"-> {len(ev):,} event-days on {len(labels)} CDT days in {YEARS}")
print(f"split per day: {((span > 1) & (span <= MAX_SPLIT_DAYS)).sum():,} multi-day "
      f"(<= {MAX_SPLIT_DAYS}d)  |  collapsed to issue day: {(span > MAX_SPLIT_DAYS).sum():,} "
      f"(> {MAX_SPLIT_DAYS}d, max {int(span.max())}d)")
print(f"flooded land cells/day: median {np.median(pos_per_day):.0f}, "
      f"max {pos_per_day.max():.0f} "
      f"({np.median(pos_per_day)/land_mask.sum():.1%} of land)")

In [ ]:
# look at the busiest flood day's label map
busy_day = max(labels, key=lambda d: labels[d].sum())
ymap = labels[busy_day]
shown = np.where(land_mask, ymap, np.nan)          # grey-out ocean for clarity

plt.figure(figsize=(10, 8))
plt.imshow(np.where(land_mask, 0.15, np.nan), cmap="Greys", vmin=0, vmax=1)
plt.imshow(shown, cmap="Reds", vmin=0, vmax=1, interpolation="none")
plt.title(f"y for {busy_day}  ->  shape {ymap.shape}, "
          f"{int(ymap.sum())} flooded cells")
plt.xlabel("col"); plt.ylabel("row"); plt.tight_layout()

## 3. Input — GOES ABI bands

GOES files are one NetCDF per scan (`MCMIPC`, all 16 ABI bands inside) on `/mnt/disk4`,
one folder per **UTC** day with 8 frames (00, 03, …, 21 UTC). We keep the frames belonging
to a **CDT day** — which straddle two UTC folders (06–21 UTC of day D−1 + 00 & 03 UTC of
day D) — and the **bands in `config.BANDS`** (all emissive IR, so they read a real signal
at night too). Below: the filename→time helpers, then every selected band's raw values for
one frame.

In [ ]:
def _scan_token(p):
    "Return the s-token (s{YYYYDDDHHMM...}) from a GOES filename."
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _scan_dt(p):
    "Scan-start datetime parsed from the filename token (UTC)."
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


def goes_files(d):
    "Sorted list of the UTC day's GOES NetCDFs (GOES16 or GOES19 auto-globbed)."
    pat = f"*/{d.year}/{d.month:02d}/{d.day:02d}/*.nc"
    return sorted(DATA_DIR.glob(pat), key=_scan_token)


def goes_cdt_day(cdt_day):
    """GOES frames whose CDT date (scan UTC − 5 h) equals cdt_day, in scan order.

    A CDT day spans UTC [D 05:00, D+1 05:00), so with full-day imagery its 8 frames live
    in TWO UTC folders: the 06–21 UTC frames in folder `cdt_day` plus the 00 & 03 UTC
    frames in folder `cdt_day + 1`. Glob both and keep the frames whose CDT date matches.
    """
    nxt = cdt_day + timedelta(days=1)
    cand = sorted(goes_files(cdt_day) + goes_files(nxt), key=_scan_token)
    return [f for f in cand if (_scan_dt(f) - timedelta(hours=5)).date() == cdt_day]


# pick one example day whose CDT D-1 has all 8 frames (a real sample)
example_label_day = sorted(d for d in labels
                           if len(goes_cdt_day(d - timedelta(days=1))) == T_FRAMES)[5]
example_in_day = example_label_day - timedelta(days=1)
files = goes_cdt_day(example_in_day)
print(f"CDT D-1 input {example_in_day}  ->  flood day {example_label_day}")
print(f"{len(files)} frames; scan times UTC:",
      [f"{_scan_dt(f):%H:%M}" for f in files])
print("frames' CDT dates (all == D-1):",
      sorted({(_scan_dt(f) - timedelta(hours=5)).date() for f in files}))

In [ ]:
# read ALL selected bands from ONE frame, to see each band's raw values
mid = files[len(files) // 2]                      # a midday-ish frame
# band -> (short name, units, colormap); every selected band is emissive IR (K)
BAND_INFO = {
    8:  ("upper WV 6.2um",    "brightness T (K)", "BuPu"),       # upper-trop water vapour
    10: ("low WV 7.3um",      "brightness T (K)", "BuPu"),       # low-trop water vapour
    11: ("cloud phase 8.4um", "brightness T (K)", "cividis"),    # cloud-top phase
    13: ("clean IR 10.3um",   "brightness T (K)", "RdYlBu_r"),   # (ref) clean IR window
    14: ("IR window 11.2um",  "brightness T (K)", "RdYlBu_r"),   # longwave IR window
    15: ("dirty IR 12.3um",   "brightness T (K)", "Spectral_r"), # dirty longwave window
    16: ("CO2 13.3um",        "brightness T (K)", "Spectral_r"), # (ref) CO2 longwave
}

with netCDF4.Dataset(mid) as nc:
    raws = {b: np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan) for b in BANDS}

ncols = 3
nrows = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.1 * nrows))
for ax, b in zip(axes.flat, BANDS):
    name, units, cmap = BAND_INFO[b]
    a = raws[b]
    im = ax.imshow(a[::4, ::4], cmap=cmap)       # subsample just for display
    ax.set_title(f"band {b} — {name}", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.85, label=units)
    ax.set_xticks([]); ax.set_yticks([])
    print(f"band {b:>2} ({name:<16}): min {np.nanmin(a):7.2f}  max {np.nanmax(a):7.2f}"
          f"  [{units}]")
for ax in axes.flat[len(BANDS):]:                # hide any unused panels
    ax.axis("off")
fig.suptitle(f"all {len(BANDS)} selected GOES bands  {example_in_day} "
             f"{_scan_dt(mid):%H:%M}Z", fontsize=11)
plt.tight_layout()

### Per-band normalization

All five bands are emissive IR brightness temperatures (~180–315 K) but sit on slightly
different scales, and the full-day cadence means each band swings over its diurnal cycle.
Standardize each band to ~zero-mean / unit-variance using per-band statistics sampled
from frames **across the whole day and both years** (cached to JSON, keyed by the year
span so it recomputes when `BANDS` / `YEARS` change). The model always sees standardized
inputs.

In [ ]:
if STATS_PATH.exists():
    stats = json.loads(STATS_PATH.read_text())
    print(f"loaded cached band stats from {STATS_PATH.name}")
else:
    # sample whole CDT days across both years; accumulate ALL frames (full diurnal range)
    cand = sorted(d - timedelta(days=1) for d in labels
                  if len(goes_cdt_day(d - timedelta(days=1))) == T_FRAMES)
    rng = np.random.default_rng(0)
    pick = rng.choice(len(cand), size=8, replace=False)
    acc = {b: [] for b in BANDS}
    for i in pick:
        for f in goes_cdt_day(cand[i])[:T_FRAMES]:        # every frame of the CDT day
            with netCDF4.Dataset(f) as nc:
                for b in BANDS:
                    a = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
                    acc[b].append(a[::4, ::4])             # subsample is plenty
    stats = {str(b): {"mean": float(np.nanmean(np.stack(acc[b]))),
                      "std": float(np.nanstd(np.stack(acc[b])))} for b in BANDS}
    STATS_PATH.parent.mkdir(parents=True, exist_ok=True)
    STATS_PATH.write_text(json.dumps(stats, indent=2))
    print(f"computed and cached band stats -> {STATS_PATH}")

BAND_MEAN = np.array([stats[str(b)]["mean"] for b in BANDS], dtype=np.float32)
BAND_STD = np.array([stats[str(b)]["std"] for b in BANDS], dtype=np.float32)
for b, m, s in zip(BANDS, BAND_MEAN, BAND_STD):
    print(f"  band {b:>2}: mean {m:8.3f}  std {s:7.3f}")

## 4. Assemble one full sample

`load_sample(in_day, label_day)` reads the 8 frames of **CDT day `in_day`** (= D−1),
normalizes each band, computes each frame's lead time (whole hours before day D 00:00 CDT =
D 05:00 UTC), and returns the `(x, t, y)` triple of the data contract. Off-Earth pixels are
zeroed after normalization.

In [ ]:
def load_sample(in_day, label_day):
    files = goes_cdt_day(in_day)[:T_FRAMES]             # CDT D-1 frames (UTC files)
    bands = np.zeros((T_FRAMES, N_BAND, IMG_H, IMG_W), dtype=np.float32)
    for t, f in enumerate(files):
        with netCDF4.Dataset(f) as nc:
            for c, b in enumerate(BANDS):
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
                bands[t, c] = (a - BAND_MEAN[c]) / BAND_STD[c]   # normalize band
    np.nan_to_num(bands, copy=False)
    # per-frame lead time: whole hours before day D's CDT start (= D 05:00 UTC)
    d05 = datetime(label_day.year, label_day.month, label_day.day, 5)
    frame_lead = np.array([round((d05 - _scan_dt(f)).total_seconds() / 3600)
                           for f in files], np.float32)
    return bands, frame_lead, labels[label_day]


t0 = time.perf_counter()
bands, frame_lead, y = load_sample(example_in_day, example_label_day)
dt = time.perf_counter() - t0
print(f"GOES seq   {bands.shape}  {bands.dtype}  = {bands.nbytes/1e6:.0f} MB")
print(f"frame_lead {frame_lead.shape}  = {frame_lead.round(1)}")
print(f"y          {y.shape}  = {int(y.sum())} flooded cells")
print(f"loaded one sample in {dt:.1f}s  (NetCDF decode)")

In [ ]:
# visualize: IR-window band across the 8 frames + the label
fig, axes = plt.subplots(1, T_FRAMES + 1, figsize=(2.0 * (T_FRAMES + 1), 2.4))
for t in range(T_FRAMES):
    axes[t].imshow(bands[t, 3, ::6, ::6], cmap="gray_r")   # channel 3 = band 14 (IR win)
    axes[t].set_title(f"-{frame_lead[t]:.0f}h", fontsize=8)
    axes[t].axis("off")
axes[-1].imshow(np.where(land_mask, 0.15, np.nan), cmap="Greys", vmin=0, vmax=1)
axes[-1].imshow(np.where(land_mask, y, np.nan), cmap="Reds", vmin=0, vmax=1)
axes[-1].set_title(f"y: {example_label_day}", fontsize=8)
axes[-1].axis("off")
fig.suptitle("one sample:  band-14 sequence (lead hrs, CDT D-1)   ->   floods (D)",
             fontsize=10)
plt.tight_layout()

## 5. Sample index + train / val / test split

A sample is valid only if **all 8 GOES frames** exist for **CDT day D−1** (i.e. both UTC
folders it spans are present). We index every valid `(D−1 → D)` pair and split it; each
sample is an independent day-to-next-day prediction.

> **Split caveat.** The split written here is random (reproducible). Adjacent days are
> weather-correlated, so a random split is slightly optimistic — notebook 02 uses a
> **temporal** split (train earlier days, test later) for the honest estimate.

In [ ]:
# SPLIT_SEED / SPLIT_FRACS come from config.py (imported above) so both notebooks agree
rows = []
for d_label in sorted(labels):
    d_in = d_label - timedelta(days=1)
    if len(goes_cdt_day(d_in)) == T_FRAMES:             # all 8 CDT D-1 frames present
        rows.append({"in_day": pd.Timestamp(d_in),
                     "label_day": pd.Timestamp(d_label),
                     "n_pos": int(labels[d_label].sum())})

index = pd.DataFrame(rows)

# reproducible random 70/20/10 assignment
rng = np.random.default_rng(SPLIT_SEED)
perm = rng.permutation(len(index))
n_tr = int(SPLIT_FRACS[0] * len(index))
n_val = int(SPLIT_FRACS[1] * len(index))
split = np.array(["test"] * len(index), dtype=object)
split[perm[:n_tr]] = "train"
split[perm[n_tr:n_tr + n_val]] = "val"
index["split"] = split

print(index["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print(f"\ntotal samples: {len(index)}  (years {YEARS})")

# class balance over the train split -> pos_weight for notebook 02's loss
tr = index[index["split"] == "train"]
tr_y = np.stack([labels[d.date()] for d in tr["label_day"]])[:, land_mask]
pos_rate = float(tr_y.mean())
print(f"train positive rate: {pos_rate:.3%}  "
      f"-> suggested pos_weight ~= {min((1-pos_rate)/pos_rate, 100):.0f}")
index.head()

## 6. Materialize the cache

Reading 8 NetCDFs per sample costs seconds *per epoch*. Pay it once: precompute every
sample and write `x` (float16, ~half size), the per-frame lead times, and the label as
separate `.npy` files, parallelized across CPU cores. Notebook 02 then memory-maps these
and trains ~10× faster. Set `BUILD_CACHE=True` to (re)build all samples — **rebuild after
any change to `BANDS`, `YEARS`, or the label source.**

> `_t.npy` (lead time) is **always** written, so the lead-time input can be toggled in
> notebook 02 via `config.USE_TIME` **without** re-decoding the GOES `x`.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE_DIR.mkdir(parents=True, exist_ok=True)
index.to_parquet(CACHE_DIR / "manifest.parquet")        # splits for notebook 02
print(f"wrote manifest ({len(index)} samples) -> {CACHE_DIR}")
# (the grid + land mask are regenerated from config.build_grid_cells(), not stored)

BUILD_CACHE = True                  # <- build ALL samples (set False for a 2-sample smoke)
# /mnt/disk4 is a single 7200-rpm HDD: the decode is read-bandwidth bound (~195 MB/s),
# so it saturates at ~12 workers — more workers also help the glob-bound label/lead refresh
# (decode stays HDD-capped; benchmark 12=24=48 equal; 32 workers per request).
N_WORKERS = 32


def _write_one(args):
    in_day, label_day = args
    stem = CACHE_DIR / f"{label_day:%Y%m%d}"
    files = goes_cdt_day(in_day)[:T_FRAMES]               # CDT D-1 frames
    # labels (sjoin lookup) + lead-times (filename parse) are cheap and depend on
    # LABEL_SOURCES / the CDT anchor -> always refresh. Only the GOES x decode is slow.
    np.save(f"{stem}_y.npy", labels[label_day].astype(np.uint8))
    d05 = datetime(label_day.year, label_day.month, label_day.day, 5)   # D 00:00 CDT
    frame_lead = np.array([round((d05 - _scan_dt(f)).total_seconds() / 3600)
                           for f in files], np.float32)    # whole hours before D start
    np.save(f"{stem}_t.npy", frame_lead)
    if Path(f"{stem}_x.npy").exists():
        return 0                                            # GOES x already cached
    bands, _, _ = load_sample(in_day, label_day)
    np.save(f"{stem}_x.npy", bands.astype(np.float16))      # GOES sequence (decode)
    return bands.nbytes // 2


todo = list(zip(index["in_day"].dt.date, index["label_day"].dt.date))
if not BUILD_CACHE:
    todo = todo[:2]
    print(f"smoke test: writing {len(todo)} samples "
          "(set BUILD_CACHE=True for all)")
else:
    gb = len(todo) * bands.nbytes / 2 / 1e9
    print(f"building ALL {len(todo)} samples (labels + lead-times refreshed; GOES x "
          f"reused where cached, else ~{gb:.0f} GB f16, ~25 min HDD-bound) "
          f"on {N_WORKERS} workers ...")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    written = list(ex.map(_write_one, todo))
print(f"done: {sum(b > 0 for b in written)} GOES seqs decoded, "
      f"{len(todo)} label+lead maps written in "
      f"{(time.perf_counter()-t0)/60:.1f} min  -> {CACHE_DIR}")

---
### Data contract recap

The cache under `cache/floodnet_2019_2020/` holds, per sample day:
- `{date}_x.npy` `(8, 5, 1500, 2500)` f16 — 8 frames of CDT day D−1 × `BANDS` (emissive IR)
- `{date}_t.npy` `(8,)` f32 — per-frame lead times (hours before D 00:00 CDT = D 05:00 UTC)
- `{date}_y.npy` `(GRID_R, GRID_C)` uint8 — observed-flood map on the 50 km grid

plus `manifest.parquet` (the split). The grid + land mask are regenerated from
`config.build_grid_cells()`, never stored.

**Next:** `model_results.ipynb` trains & compares the GOES model suite
(R(2+1)D / 3D-CNN / ConvLSTM / U-Net) against the climatology baseline.